In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from algbench import read_as_pandas
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

sns.set_theme()

In [ ]:
problematic_instances = set()


def read_row(row):
    global problematic_instances
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = (
        AnnotatedSolution.model_validate_json(row["result"]["solution"])
        if row["result"]["solution"]
        else None
    )

    if solution is None:
        problematic_instances.add(
            (
                row["parameters"]["args"]["kwargs"]["instance_name"],
                row["parameters"]["args"]["alg_params"]["mip_gap"],
            )
        )

        return None
    warmstart_solution = row["result"]["warmstart_upper_bound"]

    # if row["parameters"]["args"]["alg_params"]["root"] == "LongestPair":
    #    return None

    instance_name = row["parameters"]["args"]["kwargs"]["instance_name"]

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "is_optimal": solution.is_optimal,
        "relative_gap": solution.relative_gap,
        "instance_name": instance_name,
        "instance": instance,
        "instance_type": instance.derive_instance_type(),
        "solve_time": row["result"]["mip_time"],
        "warmstart_ub": row["result"]["warmstart_upper_bound"],
        "mip_time": row["result"]["mip_time"],
        "n": instance.num_polygons(),
        "eps": row["parameters"]["args"]["alg_params"]["mip_gap"],
    }


benchmark = read_as_pandas("results_mip_nonconvex_300", read_row)
print("loaded", len(benchmark), "runs")
assert len(problematic_instances) == 0

In [ ]:
len(benchmark[(~benchmark["is_optimal"]) & (benchmark["eps"] == 0.15)]["instance_name"])

In [ ]:
fig, ax = plt.subplots()

benchmark_rounded_n = benchmark.copy()
# benchmark_rounded_n["n"] = benchmark_rounded_n["n"].apply(lambda x: math.ceil(x / 10.0) * 10)

sns.boxplot(x="n", y="solve_time", hue="eps", data=benchmark_rounded_n, palette="tab10")

In [ ]:
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE

fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)

earliest_all = float("inf")
df = {
    "solve_time": [],
    "num_solved": [],
    "eps": [],
}

print(benchmark["instance_name"].nunique())

last_solved_for_algo = dict.fromkeys(benchmark["eps"].unique(), 0)

for _, row in benchmark.sort_values(by=["solve_time"], ascending=True).iterrows():
    if not row["is_optimal"]:
        continue

    key = row["eps"]
    last_solved_for_algo[key] += 1
    df["solve_time"].append(row["solve_time"])
    df["num_solved"].append(last_solved_for_algo[key])
    df["eps"].append(row["eps"])

print(earliest_all)

df = pd.DataFrame(df)

sns.lineplot(
    df, ax=ax, x="solve_time", y="num_solved", hue="eps", drawstyle="steps-pre", palette="tab10"
)

ax.set_ylabel("\\#solved instances")
ax.set_xlabel("solve time [s]")
# ax.set_ylim(300, 600)
ax.axvline(earliest_all, color="black", linestyle="dashed")

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, ncols=3)

fig.savefig("../plots/rq5_mip/optimality.pdf", bbox_inches="tight")

In [ ]:
benchmark_best = benchmark[(benchmark["eps"] == 0.01) & (benchmark["is_optimal"])].copy()

benchmark_best["ratio_warmstart"] = benchmark_best["warmstart_ub"] / benchmark_best["upper_bound"]
fig, ax = plt.subplots()

sns.boxplot(x="n", y="ratio_warmstart", data=benchmark_best, palette="tab10")

In [ ]:
map_root_name = {
    "LongestEdgePlusFurthestSite": "LEFP",
    "RandomRoot": "Random",
    "OrderRoot": "CHR",
    "LongestPair": "LE",
}


def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = (
        AnnotatedSolution.model_validate_json(row["result"]["solution"])
        if row["result"]["solution"]
        else None
    )

    if len(instance.polygons) > 20:
        return None

    if solution is None:
        print(
            row["result"]["error"],
            row["parameters"]["args"]["kwargs"]["instance_name"],
            row["parameters"]["args"]["alg_params"],
        )
        return None

    # if row["parameters"]["args"]["alg_params"]["root"] == "LongestPair":
    #    return None

    instance_name = row["parameters"]["args"]["kwargs"]["instance_name"]

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "is_optimal": solution.is_optimal,
        "relative_gap": solution.relative_gap,
        "instance_name": instance_name,
        "instance": instance,
        "instance_type": instance.derive_instance_type(),
        "solve_time": row["result"].get("solve_time"),
        "n": instance.num_polygons(),
        "eps": row["parameters"]["args"]["alg_params"]["eps"],
        "root": map_root_name[row["parameters"]["args"]["alg_params"]["root"]],
    }


baseline_benchmark = read_as_pandas("../05_optimality_gaps/results_optimality_gaps_300", read_row)
print("loaded", len(baseline_benchmark), "runs")

In [ ]:
print(
    len(
        set(benchmark["instance_name"].unique()) & set(baseline_benchmark["instance_name"].unique())
    )
)
print(len(set(benchmark["instance_name"].unique())))
print(len(set(baseline_benchmark["instance_name"].unique())))
print(len(baseline_benchmark["eps"].unique()))
print(baseline_benchmark["root"].unique())

In [ ]:
df = {"solve_time": [], "num_solved": [], "eps": [], "algorithm": []}

last_solved_for_algo = {
    (alg, eps): 0 for eps in benchmark["eps"].unique() for alg in ["IP", "LEFP", "CHR"]
}

for _, row in benchmark.sort_values(by=["solve_time"], ascending=True).iterrows():
    if not row["is_optimal"]:
        continue

    key = ("IP", row["eps"])
    last_solved_for_algo[key] += 1
    df["solve_time"].append(row["solve_time"])
    df["num_solved"].append(last_solved_for_algo[key])
    df["eps"].append(row["eps"])
    df["algorithm"].append("IP")


for _, row in (
    baseline_benchmark[baseline_benchmark["root"] == "CHR"]
    .sort_values(by=["solve_time"], ascending=True)
    .iterrows()
):
    if not row["is_optimal"]:
        continue

    key = (row["root"], row["eps"])
    last_solved_for_algo[key] += 1
    df["solve_time"].append(row["solve_time"])
    df["num_solved"].append(last_solved_for_algo[key])
    df["eps"].append(row["eps"])
    df["algorithm"].append(row["root"])


df = pd.DataFrame(df)

In [ ]:
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE

fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)
sns.lineplot(
    df,
    ax=ax,
    x="solve_time",
    y="num_solved",
    style="algorithm",
    hue="eps",
    drawstyle="steps-pre",
    palette="tab10",
)

ax.set_ylabel("\\#solved instances")
ax.set_xlabel("solve time [s]")
# ax.set_ylim(300, 600)
ax.axvline(earliest_all, color="black", linestyle="dashed")

handles, labels = ax.get_legend_handles_labels()
ax.legend().remove()
fig.legend(handles=handles, labels=labels, ncols=3, loc="upper right")

fig.savefig("../plots/rq5_mip/optimality_comparison.pdf", bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(ncols=5, figsize=(12, 3), sharey=True)

for eps, ax in zip(benchmark["eps"].unique(), axs):
    filtered = benchmark[benchmark["eps"] == eps]
    sns.countplot(x="is_optimal", data=filtered, hue="n", ax=ax, palette="tab20")
    ax.set_title(f"target gap={eps}")
    labels, handles = ax.get_legend_handles_labels()
    ax.legend().remove()

fig.legend(labels, handles, ncols=5, loc="upper center", title="n")
fig.subplots_adjust(top=0.6)

In [ ]:
# validate that solutions are correct.
unified_benchmark = pd.concat([benchmark, baseline_benchmark])
for eps in unified_benchmark["eps"].unique():
    n_checks = 0
    current_bench = unified_benchmark[unified_benchmark["eps"] == eps]
    for _, row in current_bench.iterrows():
        solution = row["solution"]
        if solution is None:
            continue
        assert solution.check_feasibility(row["instance"], eps=0.01)

        same_instance = current_bench[current_bench["instance_name"] == row["instance_name"]]
        for _, other in same_instance.iterrows():
            if other["solution"] is None:
                continue
            check = solution.plausibility_check(
                other["solution"],
                eps=eps + 0.001,
            )
            assert check["valid"], f"Check failed for {row['instance_name']}, {check}"
            n_checks += 1
    print(n_checks, "checks passed for eps=", eps)